In [0]:
from pyspark.sql.functions import *

silver_df = spark.read.table(
    "workspace.silver.earthquakes"
)
#display(silver_df)


In [0]:
#del dim_location

In [0]:

dim_location = (
    silver_df
    .select(
        col("latitude"),
        col("longitude")
    )
    .dropDuplicates()
)

In [0]:
import requests
import time

def obtener_pais(lat, lon):

    url = "https://nominatim.openstreetmap.org/reverse"

    params = {
        "lat": lat,
        "lon": lon,
        "format": "json"
    }

    headers = {
        "User-Agent": "earthquake-project"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    data = response.json()

    address = data.get("address", {})

    return (
        address.get("country"),
        address.get("state"),
        address.get("city")
        or address.get("town")
        or address.get("village")
    )

In [0]:
ubicaciones = []

for row in dim_location.collect():

    country, state, city = obtener_pais(
        row.latitude,
        row.longitude
    )

    ubicaciones.append(
        (
            row.latitude,
            row.longitude,
            country,
            state,
            city
        )
    )

    time.sleep(1)

In [0]:
dim_location = spark.createDataFrame(
    ubicaciones,
    [        
        "latitude",
        "longitude",
        "country",
        "state",
        "city"
    ]
)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_location = (
    dim_location
    .withColumn(
        "location_key",
        monotonically_increasing_id()
    )
)

In [0]:
dim_location.show()

In [0]:
from pyspark.sql.functions import col, coalesce, lit

dim_location = (
    dim_location
    .withColumn(
        "country",
        coalesce(col("country"), lit("Unknown"))
    )
    .withColumn(
        "state",
        coalesce(col("state"), lit("Unknown"))
    )
    .withColumn(
        "city",
        coalesce(col("city"), lit("Unknown"))
    )
)

display(dim_location)

In [0]:
#from pyspark.sql.functions import monotonically_increasing_id

#dim_location = (
#    dim_location
#    .withColumn(
#        "location_key",
#        monotonically_increasing_id()
#    )
#)

In [0]:
dim_location = dim_location.select(
    "location_key",
    "country",
    "state",
    "city",
    "latitude",
    "longitude"
)

#display(dim_location)

In [0]:
#%sql
#DROP TABLE IF EXISTS workspace.gold.dim_location;

In [0]:
(
    dim_location.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.dim_location"
    )
)

print("dim_location creada")